# 🎓 Latih AI penguji VOIC (Colab gratis)

**Guru:** Kimi K3 — sudah membuat contoh percakapan ujian lewat `php artisan voic:teacher-generate` di laptop.
**Murid:** Qwen3-4B — dilatih di sini meniru gaya Kimi K3, lalu dijalankan **100% lokal di laptop lewat Ollama (CPU only)**.

Jalankan sel dari atas ke bawah (tombol ▶ di kiri setiap sel). Total waktu ± 30–60 menit.

| Langkah | Di mana |
|---|---|
| 1. Buat data dengan Kimi K3 | Laptop: `php artisan voic:teacher-generate` lalu `php artisan voic:export-training-data` |
| 2. Latih murid | **Notebook ini** |
| 3. Pasang di Ollama | Laptop: `ollama create voic-coach -f Modelfile` |

## Langkah 0 — Nyalakan GPU gratis
Menu **Runtime → Change runtime type → T4 GPU → Save**. Lalu jalankan sel di bawah: harus muncul tulisan **Tesla T4**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Langkah 1 — Pasang alat training (± 3 menit)

In [ ]:
%%capture
!pip install -q unsloth

## Langkah 2 — Unggah 3 file dari laptop
Klik **Choose Files**, lalu pilih ketiga file ini dari folder proyek VOIC:
- `ai_service/datasets/sft/voic_exam_sft.jsonl` (data latih)
- `ai_service/datasets/sft/voic_exam_eval.jsonl` (data uji)
- `ai_service/finetune_lora.py` (skrip training)

In [ ]:
import os, shutil
from google.colab import files

uploaded = files.upload()
os.makedirs("datasets/sft", exist_ok=True)
for name in uploaded:
    if name.endswith(".jsonl"):
        shutil.move(name, os.path.join("datasets/sft", name))

for path in ["datasets/sft/voic_exam_sft.jsonl", "datasets/sft/voic_exam_eval.jsonl", "finetune_lora.py"]:
    if os.path.exists(path):
        lines = sum(1 for _ in open(path, encoding="utf-8")) if path.endswith(".jsonl") else None
        print("✓", path, f"({lines} contoh)" if lines is not None else "")
    else:
        print("✗ belum diunggah:", path)

## Langkah 3 — Latih murid (± 20–40 menit)
- Data < 200 contoh: biarkan `--epochs 2` agar tidak "menghafal".
- Data > 500 contoh: boleh `--epochs 3`.

Di akhir, skrip otomatis mengubah model menjadi file **GGUF** (format Ollama) + **Modelfile**.

In [ ]:
!python finetune_lora.py \
    --train datasets/sft/voic_exam_sft.jsonl \
    --eval datasets/sft/voic_exam_eval.jsonl \
    --base unsloth/Qwen3-4B-Instruct-2507 \
    --epochs 2 \
    --output outputs/voic-coach

## Langkah 4 — Simpan hasil ke Google Drive (disarankan)
File GGUF ± 2,5 GB. Menyimpan ke Drive lebih aman daripada unduh langsung (unduhan besar di browser sering putus).
Setelah selesai, buka Google Drive → folder **VOIC-model**, unduh **file .gguf** dan **Modelfile** ke satu folder yang sama di laptop.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

target = "/content/drive/MyDrive/VOIC-model"
os.makedirs(target, exist_ok=True)
for name in os.listdir("outputs/voic-coach"):
    if name.endswith(".gguf") or name == "Modelfile":
        shutil.copy(os.path.join("outputs/voic-coach", name), target)
        print("✓ disalin:", name)

## Langkah 5 — Pasang di laptop (tanpa internet, CPU only)
Buka Command Prompt di folder tempat file **.gguf** dan **Modelfile** berada, lalu:

```bash
ollama create voic-coach -f Modelfile
```

Pastikan di file `.env` Laravel baris `VOIC_AI_MODEL=` **kosong** (agar memakai model lokal `voic-coach`), lalu nyalakan ulang `jalankan_voic.bat`.

Bandingkan hasilnya (bukti untuk dosen):

```bash
cd ai_service
python eval_dialogue.py --models voic-coach qwen3:4b
```